# Collect → Train → Deploy with SO-101

End-to-end workflow: collect teleoperation demos on an **SO-101** arm, fine-tune a **π0.5** policy, and deploy it back on the robot using `physicalai` runtime.

1. **Collect** demonstration data (teleoperation with leader/follower arms)
2. **Train** a π0.5 visuomotor diffusion policy
3. **Deploy** the trained policy on hardware via `physicalai` runtime

---

## Prerequisites

### What you need

| | Requirement |
|--|-------------|
| **Robot** | SO-101 follower arm (+ leader arm for data collection) |
| **Cameras** | 1–2 USB cameras (UVC/RealSense/Basler compatible) |
| **Training GPU** | 40 GB+ VRAM (e.g. 2*B70 or A100) |
| **Deploy machine** | Intel CPU, GPU, or NPU with OpenVINO support |
| **OS** | Linux (Ubuntu 22.04+ recommended) |
| **Python** | 3.12 or newer |

### Install packages

- **`physicalai`** — runtime for deployment (Step 3)
- **`physicalai-train`** — fine-tuning library (Step 2), distributed via [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio)

In [ ]:
%pip install physicalai[so101,capture] physicalai-train

## Step 1: Data Collection

Collect teleoperated demonstrations of the task you want the robot to learn.

### Option A: Physical AI Studio (recommended)

Use the [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) web UI for guided data collection with a leader-follower setup. See the repository README for installation and setup instructions.

The web UI guides you through:
- Configuring leader/follower arms and cameras
- Recording episodes via teleoperation
- Reviewing and curating collected episodes

### Option B: LeRobot directly

Use the [LeRobot](https://github.com/huggingface/lerobot) library for programmatic data collection:

```bash
pip install lerobot

python lerobot/scripts/control_robot.py \
  --robot.type=so100 \
  --control.type=record \
  --control.fps=30 \
  --control.repo_id=your_org/pick_can_so100 \
  --control.num_episodes=50 \
  --control.single_task="Pick up the can" \
  --control.push_to_hub=True
```

> **Note:** LeRobot uses `so100` as the robot type for both SO-100 and SO-101 hardware.

Aim for **50–100 demonstrations** of the target task with varied object positions.

## Step 2: Train a Policy

Train a π0.5 (pi-zero-point-five) visuomotor diffusion policy on the collected demonstrations.

### Option A: Physical AI Studio (recommended)

Launch a training job from the web UI — select your dataset, choose the π0.5 architecture, and configure training parameters visually.

Recommended training parameters:
| Parameter | Value |
|-----------|-------|
| Batch size | auto |
| Steps | 50 000 |
| Data workers | auto |
| Precision | BF16 mixed |
| Compile model | true |


> **Note:** Training π0.5 requires at least **40 GB of VRAM** (e.g. NVIDIA A100).

### Option B: `physicalai-train` library

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

from physicalai.data import LeRobotDataModule
from physicalai.policies import Pi05
from physicalai.train import Trainer

# ── Config ──
# Physical AI Studio: ~/.cache/physicalai/datasets/<dataset-id>/<dataset-name>
# LeRobot:            data/<repo_id>  (relative to where you ran the collection script)
DATASET_PATH = "~/.cache/physicalai/datasets/<dataset-id>/<dataset-name>"
CHECKPOINT_DIR = "./checkpoints"

# Fine-tune π0.5 from the pretrained base
model = Pi05(
    pretrained_name_or_path="lerobot/pi05_base",
    dtype="bfloat16",
    gradient_checkpointing=True,
)

datamodule = LeRobotDataModule(
    root=DATASET_PATH,
    train_batch_size=8,
    data_format="physicalai",
    num_workers=8,
    val_split=0.1,
)

checkpoint_cb = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    monitor="val/loss",
    mode="min",
    save_top_k=1,
    save_last=True,
)

trainer = Trainer(
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    precision="bf16-mixed",
    log_every_n_steps=20,
    check_val_every_n_epoch=1,
    callbacks=[checkpoint_cb],
)

trainer.fit(model=model, datamodule=datamodule)

print(f"Best checkpoint: {checkpoint_cb.best_model_path}")

### Export to OpenVINO

If using **Physical AI Studio**, the model is automatically exported to OpenVINO format when you download it — no extra step needed.

If training via the library, export manually:

In [ ]:
# Load best checkpoint and export to OpenVINO IR
best_model = Pi05.load_from_checkpoint(checkpoint_cb.best_model_path, compile_model=False)
best_model.eval()
best_model.to_openvino("./exports/my_policy_openvino")

---

## Step 3: Deploy on Hardware

Run the trained policy on the physical robot using `physicalai` runtime. This section will:
1. Discover connected cameras
2. Configure hardware and model paths
3. Load the exported policy
4. Connect cameras and robot
5. Run a 60-second policy episode

### Configuration

In [ ]:
# Discover available cameras — use the device IDs from this output in the config below
from physicalai.capture import discover_all

for driver, devices in discover_all().items():
    if not devices:
        continue
    print(f"\n[{driver}]")
    for dev in devices:
        print(f"  {dev.device_id}  —  {dev.name}")

In [ ]:
# --- Hardware & Model Config ---
EXPORT_DIR = "exports/my_policy_openvino"
DEVICE = "GPU"  # OpenVINO device: "GPU", "CPU", "NPU"
TASK = "pick up the can"

# Robot
SO101_PORT = "/dev/ttyACM0"
# Calibration file location depends on how you collected data:
#   Physical AI Studio: ~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json
#   LeRobot:            ~/.cache/calibration/so101_follower.json
SO101_CALIBRATION = "~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json"

# Cameras — use the same names and setup as during dataset collection
CAMERAS = [
    ("overhead", "uvc", "/dev/video0"),
    ("arm", "uvc", "/dev/video1"),
]
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_FPS = 30

# Runtime
FPS = 30
DURATION_S = 60.0

### Load the exported model

In [ ]:
import openvino_tokenizers  # noqa: F401 — registers OV tokenizer ops
from physicalai.inference import InferenceModel

policy = InferenceModel.load(EXPORT_DIR, device=DEVICE)
print(f"Model loaded on {DEVICE}")

### Connect cameras and robot

In [ ]:
from typing import Any

from physicalai.capture import SharedCamera
from physicalai.robot import SO101

# Cameras
cameras = {}
for name, driver, device_id in CAMERAS:
    kwargs: dict[str, Any] = {"serial_number": device_id} if driver == "realsense" else {"device": device_id}
    cam = SharedCamera(driver, **kwargs, width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS)
    cam.connect()
    cameras[name] = cam
    print(f"Camera '{name}' connected: {cam.actual_width}x{cam.actual_height} @ {cam.actual_fps}fps")

# Robot
robot = SO101(port=SO101_PORT, calibration=SO101_CALIBRATION, role="follower")
robot.connect()
print(f"Robot connected on {SO101_PORT}")

### Build and run the policy runtime

The `PolicyRuntime` handles the control loop: observe → infer → send actions at the target FPS.

> **Safety:** The robot will begin moving immediately. Keep your hand near the power switch or e-stop.

In [ ]:
from physicalai.runtime import ChunkedActionQueue, PolicyRuntime, SyncExecution

runtime = PolicyRuntime(
    robot=robot,
    model=policy,
    execution=SyncExecution(request_threshold=0.5),
    fps=FPS,
    cameras=cameras,
    action_queue=ChunkedActionQueue(),
    task=TASK,
)

with runtime:
    print(f"Running policy at {FPS} fps for {DURATION_S}s — task: {TASK!r}")
    stats = runtime.run(duration_s=DURATION_S)

print(f"\nDone — {stats.steps} steps, {stats.inference_count} inferences, {stats.total_holds} holds")

### Disconnect

In [ ]:
for name, cam in cameras.items():
    cam.disconnect()
    print(f"Camera '{name}' disconnected")

robot.disconnect()
print("Robot disconnected")